In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import dpctl
from sklearnex import patch_sklearn, config_context
patch_sklearn()

In [ ]:
from sklearn.manifold import TSNE
from sklearn.cluster import DBSCAN

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
from matplotlib.pyplot import ylim, xlim

In [ ]:
import xgboost as xgb

from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import gc
import shap
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def gini(actual, pred, cmpcol = 0, sortcol = 1):
    assert( len(actual) == len(pred) )
    all = np.asarray(np.c_[ actual, pred, np.arange(len(actual)) ], dtype=float)
    all = all[ np.lexsort((all[:,2], -1*all[:,1])) ]
    totalLosses = all[:,0].sum()
    giniSum = all[:,0].cumsum().sum() / totalLosses
    
    giniSum -= (len(actual) + 1) / 2.
    return giniSum / len(actual)
 
def gini_normalized(a, p):
    return gini(a, p) / gini(a, a)

In [ ]:
train = pd.read_csv('../input/porto-seguro-safe-driver-prediction/train.csv')
test = pd.read_csv('../input/porto-seguro-safe-driver-prediction/test.csv')
sample_submission = pd.read_csv('../input/porto-seguro-safe-driver-prediction/sample_submission.csv')

In [ ]:
features = train.columns[2:]

In [ ]:
features

In [ ]:
faulty_columns = ['ps_ind_10_bin', 'ps_ind_11_bin', 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_ind_14', 'ps_car_10_cat']

In [ ]:
new_features = list(filter(lambda x: x not in faulty_columns, features))

In [ ]:
train['ps_ind_03-ps_ind_02_cat'] = train['ps_ind_03']*train['ps_ind_02_cat']
train['ps_car_13-ps_ind_03'] = train['ps_car_13']*train['ps_ind_03']

test['ps_ind_03-ps_ind_02_cat'] = test['ps_ind_03']*test['ps_ind_02_cat']
test['ps_car_13-ps_ind_03'] = test['ps_car_13']*test['ps_ind_03']

new_features += ['ps_ind_03-ps_ind_02_cat', 'ps_car_13-ps_ind_03']

In [ ]:
X = train[new_features]
X_test = test[new_features]
Y = train.target.values

In [ ]:
params = {'objective': 'binary:logistic',
          'tree_method': 'hist',
          'device': 'cuda',
          'lambda': 4.645511,
 'alpha': 0.654147,
 'colsample_bytree': 0.917,
 'subsample': 0.66,
 'learning_rate': 0.013,
 'max_depth': 7,
 'min_child_weight': 194,
 'eval_metric': 'logloss'}

In [ ]:
dtest = xgb.DMatrix(X_test, enable_categorical=True)

In [ ]:
%%time
train_oof = np.zeros((X.shape[0], ))
train_oof_shap = np.zeros((X.shape[0],X.shape[1]+1))
test_preds = 0
train_oof.shape
num_round = 900

n_splits = 5
kf = KFold(n_splits=n_splits, random_state=137, shuffle=True)

for jj, (train_index, val_index) in enumerate(kf.split(X)):
        print("Fitting fold", jj+1)
        train_features = X.loc[train_index]
        train_target = Y[train_index]

        val_features = X.loc[val_index]
        val_target = Y[val_index]

        dtrain = xgb.DMatrix(train_features, train_target, enable_categorical=True)
        dval = xgb.DMatrix(val_features, val_target, enable_categorical=True)

        model = xgb.train(params, dtrain, num_round)
        #model.set_param({'predictor': 'cpu_predictor'})
        val_pred = model.predict(dval)
        temp_oof_shap = model.predict(dval, pred_contribs=True)
        
        train_oof[val_index] = val_pred
        train_oof_shap[val_index, :] = temp_oof_shap
        
        print("Fold normalized:", gini_normalized(val_target, val_pred))
        test_preds += model.predict(dtest)/n_splits
        del train_features, train_target, val_features, val_target, temp_oof_shap
        gc.collect()

In [ ]:
gini_normalized(Y, train_oof)

In [ ]:
%%time
tsne = TSNE(n_components=2)
train_2D = tsne.fit_transform(train_oof_shap)

In [ ]:
plt.scatter(train_2D[:,0], train_2D[:,1], c=Y, s = 0.8)

In [ ]:
sample_submission['target'] = test_preds
sample_submission.head()

In [ ]:
sample_submission.to_csv('submission.csv', index=False)